In [2]:
# ============================================================
# 05_joblib_pipeline.ipynb
# Joblib ML Pipeline - NYC Taxi Dataset
# ============================================================

# 1. Imports

import time
import math
import requests
import pandas as pd
import numpy as np

from joblib import Parallel, delayed
from pydantic import ByteSize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor


In [4]:
# 2. Configuration

RUN_MODE = "local"

if RUN_MODE == "local":
    DATA_PATH = "../data/raw/taxi/yellow_tripdata_2026-01.parquet"
else:
    DATA_PATH = "gs://your-bucket/taxi/yellow_tripdata_2026-01.parquet"

LOOKUP_PATH = "../data/raw/taxi/taxi_zone_lookup.csv"

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TARGET = "fare_amount"

FEATURES = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

N_JOBS = -1


In [12]:
from collections.abc import Generator
from contextlib import contextmanager
from datetime import timedelta
from pathlib import Path
from time import perf_counter
from typing import TypedDict

class Duration(TypedDict):
    duration: timedelta


def _mem_str(path: str | Path) -> str:
    path = Path(path)
    size = path.stat().st_size
    return ByteSize(size).human_readable(decimal=True)


def download_if_missing(path: str | Path, url: str) -> None:
    path = Path(path)

    if path.exists():
        print(f"{path.name} already exists, {_mem_str(path)}")
        return

    print(f"Downloading {path.name}...")

    response = requests.get(url)
    response.raise_for_status()

    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded {path}, {_mem_str(path)}")


@contextmanager
def timeit(name: str) -> Generator[Duration, None, None]:
    result: Duration = {"duration": timedelta()}
    start_time = perf_counter()

    try:
        yield result
    finally:
        duration = timedelta(seconds=perf_counter() - start_time)
        print(f"{name} took {duration}")
        result["duration"] = duration



In [6]:
# 3. Load data

download_if_missing(DATA_PATH, SOURCE_URL)
download_if_missing(LOOKUP_PATH, LOOKUP_URL)

with timeit("Loading data"):
    df = pd.read_parquet(DATA_PATH)

location_lookup = pd.read_csv(LOOKUP_PATH)

df.head()


Downloaded ../data/raw/taxi/taxi_zone_lookup.csv, 12.3KB
Loading data took 0:00:00.321791


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [13]:
# 4. Basic dataset inspection

print("Columns:")
print(df.columns)

print("\nDtypes:")
print(df.dtypes)

print("\nShape:")
print(df.shape)


Columns:
Index(['trip_distance', 'passenger_count', 'PULocationID', 'DOLocationID',
       'payment_type', 'fare_amount'],
      dtype='str')

Dtypes:
trip_distance      float64
passenger_count    float64
PULocationID         int32
DOLocationID         int32
payment_type         int64
fare_amount        float64
dtype: object

Shape:
(2551851, 6)


In [15]:
# 5. Preprocessing

df = df[FEATURES + [TARGET]]
df = df.dropna()

df = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["passenger_count"] > 0)
]

df.tail()


,trip_distance,passenger_count,PULocationID,DOLocationID,payment_type,fare_amount
2636822,2.75,2.0,166,237,1,17.0
2636827,1.86,1.0,230,239,1,12.1
2636828,2.66,1.0,239,262,1,16.3
2636829,0.36,4.0,170,170,1,5.8
2636830,1.15,1.0,230,229,1,10.7


In [17]:
def split_dataframe(df: pd.DataFrame, n_chunks: int) -> list[pd.DataFrame]:
    chunk_size = math.ceil(len(df) / n_chunks)
    return [
        df.iloc[i:i + chunk_size].copy()
        for i in range(0, len(df), chunk_size)
    ]


def benchmark_joblib(data_path: str, lookup_path: str, n_jobs: int = -1):
    metrics = {}

    # 1. Read Data
    with timeit("1. Read Data") as duration:
        df = pd.read_parquet(data_path)
        df_lookup = pd.read_csv(lookup_path)
    metrics["1. Read Data"] = duration["duration"]

    if n_jobs == -1:
        effective_jobs = 4  # safe default for chunking; Joblib still uses all cores where applicable
    else:
        effective_jobs = max(1, n_jobs)

    chunks = split_dataframe(df, effective_jobs)

    # 2. Count Operation
    with timeit("2. Count Operation") as duration:
        partial_counts = Parallel(n_jobs=n_jobs)(
            delayed(lambda chunk: chunk.count())(chunk)
            for chunk in chunks
        )
        _ = pd.concat(partial_counts, axis=1).sum(axis=1)
    metrics["2. Count Operation"] = duration["duration"]

    # 3. Complex Arithmetic Formula
    with timeit("3. Complex Arithmetic") as duration:
        partial_results = Parallel(n_jobs=n_jobs)(
            delayed(lambda chunk: (chunk["fare_amount"] + chunk["tip_amount"]) * 1.5 / (chunk["passenger_count"] + 1))(chunk)
            for chunk in chunks
        )
        _ = pd.concat(partial_results)
    metrics["3. Complex Arithmetic"] = duration["duration"]

    # 4. Statistical Standard Deviation
    with timeit("4. Standard Deviation") as duration:
        # For simplicity and correctness, calculate std directly in pandas.
        # Joblib is less natural for global reductions than Dask/PySpark.
        _ = df["trip_distance"].std()
    metrics["4. Standard Deviation"] = duration["duration"]

    # 5. GroupBy Aggregation
    with timeit("5. GroupBy Mean") as duration:
        partial_grouped = Parallel(n_jobs=n_jobs)(
            delayed(lambda chunk: chunk.groupby("passenger_count")["fare_amount"].agg(["sum", "count"]))(chunk)
            for chunk in chunks
        )

        grouped = pd.concat(partial_grouped).groupby(level=0).sum()
        grouped["mean"] = grouped["sum"] / grouped["count"]
        _ = grouped["mean"]
    metrics["5. GroupBy Mean"] = duration["duration"]

    # 6. Merge/Join with Count
    with timeit("6. Join & Count") as duration:
        df_lookup["LocationID"] = df_lookup["LocationID"].astype(df["PULocationID"].dtype)
        joined = pd.merge(df, df_lookup, left_on="PULocationID", right_on="LocationID", how="inner")
        _ = len(joined)
    metrics["6. Join & Count"] = duration["duration"]

    return metrics


In [18]:
benchmarks = benchmark_joblib(DATA_PATH, LOOKUP_PATH, n_jobs=N_JOBS)

benchmarks


1. Read Data took 0:00:00.108501
2. Count Operation took 0:00:00.871172
3. Complex Arithmetic took 0:00:00.485893
4. Standard Deviation took 0:00:00.015685
5. GroupBy Mean took 0:00:00.489243
6. Join & Count took 0:00:00.144949


{'1. Read Data': datetime.timedelta(microseconds=108501),
 '2. Count Operation': datetime.timedelta(microseconds=871172),
 '3. Complex Arithmetic': datetime.timedelta(microseconds=485893),
 '4. Standard Deviation': datetime.timedelta(microseconds=15685),
 '5. GroupBy Mean': datetime.timedelta(microseconds=489243),
 '6. Join & Count': datetime.timedelta(microseconds=144949)}

In [19]:
# 6. Regression pipeline - XGBRegressor

X = df[FEATURES]
y_reg = df[TARGET]

X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

reg_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=N_JOBS
)

with timeit("Regression training") as duration:
    reg_model.fit(X_train, y_train_reg)

regression_train_time = duration["duration"]

y_pred_reg = reg_model.predict(X_test)


Regression training took 0:00:01.657144


In [20]:
# 7. Regression metrics

regression_results = {
    "library": "Joblib",
    "task": "regression",
    "model": "XGBRegressor",
    "train_time": regression_train_time,
    "mae": mean_absolute_error(y_test_reg, y_pred_reg),
    "mse": mean_squared_error(y_test_reg, y_pred_reg),
    "rmse": mean_squared_error(y_test_reg, y_pred_reg) ** 0.5,
    "r2": r2_score(y_test_reg, y_pred_reg),
}

regression_results


{'library': 'Joblib',
 'task': 'regression',
 'model': 'XGBRegressor',
 'train_time': datetime.timedelta(seconds=1, microseconds=657144),
 'mae': 2.7971266618985595,
 'mse': 38.379493550132224,
 'rmse': 6.195118525914756,
 'r2': 0.8850842543178181}

In [21]:
# 8. Classification pipeline - LogisticRegression

def classify_fare(value):
    if value < 15:
        return 0
    elif value < 40:
        return 1
    else:
        return 2


y_cls = y_reg.apply(classify_fare)

X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

cls_model = LogisticRegression(
    max_iter=1000,
    n_jobs=N_JOBS
)

with timeit("Classification training") as duration:
    cls_model.fit(X_train, y_train_cls)

classification_train_time = duration["duration"]

y_pred_cls = cls_model.predict(X_test)


/Users/gustavooliveira/WORKSPACE - 2025-2026/FCUP/Segundo-Semestre/BigData-CloudComputing/Part2/Project/venv-bdcc/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Classification training took 0:01:42.092186


/Users/gustavooliveira/WORKSPACE - 2025-2026/FCUP/Segundo-Semestre/BigData-CloudComputing/Part2/Project/venv-bdcc/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
# 9. Classification metrics

classification_results = {
    "library": "Joblib",
    "task": "classification",
    "model": "LogisticRegression",
    "train_time": classification_train_time,
    "accuracy": accuracy_score(y_test_cls, y_pred_cls),
    "precision_macro": precision_score(y_test_cls, y_pred_cls, average="macro"),
    "recall_macro": recall_score(y_test_cls, y_pred_cls, average="macro"),
    "f1_macro": f1_score(y_test_cls, y_pred_cls, average="macro"),
}

classification_results


{'library': 'Joblib',
 'task': 'classification',
 'model': 'LogisticRegression',
 'train_time': datetime.timedelta(seconds=102, microseconds=92186),
 'accuracy': 0.8714229452692257,
 'precision_macro': 0.8905091707300982,
 'recall_macro': 0.8217799434132519,
 'f1_macro': 0.8497728396755092}

In [23]:
# 10. Save results

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

benchmark_results_df = pd.DataFrame([
    {"operation": operation, "library": "Joblib", "duration": duration}
    for operation, duration in benchmarks.items()
])

ml_results_df = pd.DataFrame([
    regression_results,
    classification_results
])

benchmark_results_df.to_csv("../results/joblib_benchmark_results.csv", index=False)
ml_results_df.to_csv("../results/joblib_pipeline_results.csv", index=False)

benchmark_results_df, ml_results_df


(               operation library               duration
 0           1. Read Data  Joblib 0 days 00:00:00.108501
 1     2. Count Operation  Joblib 0 days 00:00:00.871172
 2  3. Complex Arithmetic  Joblib 0 days 00:00:00.485893
 3  4. Standard Deviation  Joblib 0 days 00:00:00.015685
 4        5. GroupBy Mean  Joblib 0 days 00:00:00.489243
 5        6. Join & Count  Joblib 0 days 00:00:00.144949,
   library            task               model             train_time  \
 0  Joblib      regression        XGBRegressor 0 days 00:00:01.657144   
 1  Joblib  classification  LogisticRegression 0 days 00:01:42.092186   
 
         mae        mse      rmse        r2  accuracy  precision_macro  \
 0  2.797127  38.379494  6.195119  0.885084       NaN              NaN   
 1       NaN        NaN       NaN       NaN  0.871423         0.890509   
 
    recall_macro  f1_macro  
 0           NaN       NaN  
 1       0.82178  0.849773  )